# Perelman Ricci Theory to Finance — v9

## Ricci flow, singularity detection, and financial surgery

This graduate lecture notebook moves from static financial correlation networks to a Perelman-inspired pipeline: curvature → Ricci flow → singular edge detection → surgery → regime diagnostics.

## 1. Mathematical motivation

Ricci curvature describes how local neighborhoods spread or concentrate. In a financial graph, edges are built from correlation distances. Positive curvature suggests redundant, coherent local structure. Negative curvature suggests bridge-like structure, where stress can transmit between clusters.

Ricci flow evolves edge distances according to curvature. A simple graph analogue is:

$$w_{ij}^{t+1}=w_{ij}^{t}(1-\eta\kappa_{ij}^{t})$$

where $w_{ij}$ is financial distance and $\kappa_{ij}$ is Ollivier-Ricci curvature. Negative curvature stretches edges; positive curvature contracts edges.

In [ ]:
import pandas as pd
import numpy as np

from helper import (
    DEFAULT_TICKERS,
    make_demo_prices,
    prices_to_returns,
    build_rolling_frames,
    build_base_graph_for_layout,
    compute_stable_layout,
    visualize_network,
    build_plotly_animation,
    rolling_feature_table,
    plot_rolling_features,
    run_ricci_flow,
    plot_ricci_flow_history,
    compare_before_after_flow,
    perform_financial_surgery,
    compute_hmm_regimes,
    summarize_edges,
)

## 2. Create lecture data

The default uses synthetic prices so the notebook is fully runnable offline. Replace this block with `download_prices(...)` for live yfinance data.

In [ ]:
tickers = DEFAULT_TICKERS[:14]
prices = make_demo_prices(tickers=tickers, n_days=300, seed=7)
returns = prices_to_returns(prices)
prices.tail()

## 3. Build rolling Ricci financial networks

The recommended v9 graph mode is `hybrid`: threshold edges plus k-nearest-neighbor edges. This avoids an overly brittle graph where one isolated ticker creates an extra cluster merely because a hard threshold removed all of its edges.

In [ ]:
frames, starts = build_rolling_frames(
    returns,
    window_size=60,
    step=5,
    max_frames=45,
    edge_mode="hybrid",
    knn_k=3,
    max_distance=1.15,
    min_abs_corr=0.20,
    alpha=0.5,
    method="OTD",
    proc=1,
)

features = rolling_feature_table(frames)
features.tail()

In [ ]:
plot_rolling_features(features)

## 4. Animate the rolling financial manifold

The animation follows the v7 style: stable node positions, Plotly slider, and edge color by Ricci curvature.

In [ ]:
base = build_base_graph_for_layout(frames, all_nodes=returns.columns)
pos = compute_stable_layout(base, seed=42)
build_plotly_animation(frames, pos, title="Rolling Ricci Finance v9")

## 5. Inspect one frame

Choose a frame near the end of the sample and inspect edge curvature. The most negative edges are candidates for fragile bridges.

In [ ]:
fd = frames[-1]
visualize_network(fd.G, positions=pos, title=f"Selected Ricci network: {fd.stats.end_date}", node_cluster=fd.node_cluster)

In [ ]:
summarize_edges(fd.G).head(12)

## 6. Ricci flow

Ricci flow is not just another network statistic. It deforms the graph. If an edge is strongly negative curvature, its distance tends to stretch. If an edge is positive curvature, it tends to contract. This gives a geometric stress test.

In [ ]:
flowed_G, flow_history = run_ricci_flow(
    fd.G,
    iterations=10,
    step_size=0.25,
    alpha=0.5,
    method="OTD",
    proc=1,
    normalize_mean_weight=True,
)
plot_ricci_flow_history(flow_history)

In [ ]:
compare_before_after_flow(fd.G, flowed_G).head(12)

In [ ]:
visualize_network(flowed_G, positions=pos, title="After Ricci flow")

## 7. Perelman-inspired financial surgery

Classical Ricci flow may develop singularities. In Hamilton-Perelman theory, surgery cuts neck-like singular regions and continues the evolution.

In finance, v9 interprets surgery as cutting unstable contagion channels: very negative-curvature edges that are also long-distance or graph bridges.

In [ ]:
surgery = perform_financial_surgery(
    flowed_G,
    curvature_threshold=-0.35,
    distance_quantile=0.80,
    use_bridge_test=True,
    remove_isolated_nodes=False,
)

surgery.report

In [ ]:
surgery.before_stats, surgery.after_stats

In [ ]:
removed_pairs = [(u, v) for u, v, _, _ in surgery.removed_edges]
visualize_network(surgery.before, positions=pos, title="Before surgery — singular edges highlighted", highlight_edges=removed_pairs)

In [ ]:
visualize_network(surgery.after, positions=pos, title="After financial surgery")

## 8. HMM hidden regimes

The HMM is not the source of geometry. It is a latent-state summary of rolling curvature, entropy, density, and component structure. Use it as a regime classifier, then validate against returns, volatility, and news.

In [ ]:
hmm_df, regime_names = compute_hmm_regimes(
    frames,
    returns,
    starts,
    window_size=60,
    n_components=3,
    forward_days=5,
    random_state=42,
)
hmm_df.tail()

In [ ]:
hmm_df.groupby(["hmm_state", "regime_name"]).agg({
    "avg_ricci": "mean",
    "ricci_min": "mean",
    "density": "mean",
    "component_entropy": "mean",
    "date": "count",
}).rename(columns={"date": "count"})

## 9. Lecture conclusion

The v9 model is best understood as **geometric market thermodynamics**:

- curvature = local stress/coherence,
- Ricci flow = geometry evolution,
- singular edges = possible contagion necks,
- surgery = removing unstable bridges,
- entropy = fragmentation measure,
- HMM = hidden regime summarizer.

This is not a standalone trading signal. It is a research/lecture framework for studying market topology under stress.